# Week 8 — Classification for Engineering Decisions

**ESC 113: Computer Aided Analysis for Engineering**

This notebook is designed as both a **lecture guide and a self-study resource**. It explains not only how to run a classifier, but also how to interpret classification errors in an engineering context.

### Main engineering question

> Can we classify the type of surface fault on a steel plate from measured geometric and image-derived features?

We will use **multiclass logistic regression** on the UCI Steel Plates Faults dataset.

## Table of Contents

1. What Is Classification?
2. Regression vs Classification
3. Binary vs Multiclass Classification
4. Engineering Classification Examples
5. Steel Plates Faults Dataset
6. What One Row Represents
7. Target Classes
8. Feature Groups and Meanings
9. Load the Dataset
10. Inspect the Data
11. Convert Indicator Targets to One Class Label
12. Check Class Balance
13. Why Class Imbalance Matters
14. Train/Test Split
15. Why We Use Stratification
16. Why Scaling Matters
17. Logistic Regression
18. What fit() Learns
19. Make Class Predictions
20. Accuracy
21. Confusion Matrix
22. True Positive, False Positive, False Negative
23. Precision
24. Recall
25. F1 Score
26. Macro vs Weighted Averages
27. Per-Class Performance
28. Misclassified Samples
29. Predicting Probabilities
30. Confidence and Uncertain Predictions
31. Common Student Questions
32. Common Mistakes
33. Guided Practice
34. Independent Exercises
35. Review Quiz

## 1. What Is Classification?

Classification is used when the target is a **category** rather than a continuous number.

Examples:

- machine condition: Normal / Faulty;
- weld quality: Accept / Reject;
- bearing state: Healthy / Worn / Damaged;
- steel defect type;
- traffic condition: Low / Medium / High;
- sensor state: Normal / Warning / Critical.

In Week 7, we predicted a continuous value in MW. That was **regression**.

In Week 8, we predict one of several fault classes. That is **classification**.

## 2. Regression vs Classification

| Problem | Target example | Learning type |
|---|---|---|
| Predict power output | 453.8 MW | Regression |
| Predict concrete strength | 42.5 MPa | Regression |
| Predict fault type | Scratches | Classification |
| Predict machine state | Healthy | Classification |

### Student question: What if the classes are written as numbers?

A target stored as 0, 1, or 2 can still be categorical.

What matters is the **meaning** of the target, not only its storage type.

## 3. Binary vs Multiclass Classification

**Binary classification** has two classes.

Example:
- Healthy
- Faulty

**Multiclass classification** has more than two classes.

The steel dataset has **seven possible fault classes**, so this is a multiclass problem.

## 4. Engineering Classification Applications

Classification is common in:

- predictive maintenance;
- automated visual inspection;
- manufacturing quality control;
- structural-health monitoring;
- fault diagnosis;
- equipment alarms;
- process-state identification.

The engineering cost of a wrong prediction may differ dramatically by error type.

## 5. Dataset — Steel Plates Faults

**Source:** UCI Machine Learning Repository  
**Dataset ID:** 198  
**Observations:** 1,941  
**Input features:** 27  
**Target classes:** 7  
**Application:** automated classification of steel-plate surface defects

Each observation describes characteristics of a detected steel-plate fault.

The original dataset stores the class using seven indicator columns.

**Reference:** UCI Steel Plates Faults  
https://archive.ics.uci.edu/dataset/198/steel+plates+faults

## 6. What Does One Row Represent?

One row represents **one detected steel-plate fault instance**.

The row contains numerical descriptors of the fault, such as:

- location;
- dimensions;
- pixel-area information;
- gray-level statistics;
- geometric relationships;
- other image-derived measurements.

The target indicates the fault category.

### Student question: Is each row a complete steel plate?

No. Each row represents a detected fault instance, not an entire manufacturing batch.

## 7. Target Classes

The seven target categories are commonly represented as:

- Pastry
- Z_Scratch
- K_Scatch
- Stains
- Dirtiness
- Bumps
- Other_Faults

The original data use one indicator column per class.

For one observation, one class indicator should be active.

## 8. Feature Names and Meaning

The dataset contains 27 input features.

Some are directly interpretable, while others are image-derived engineering descriptors.

Important examples include:

| Feature | Interpretation |
|---|---|
| X_Minimum | Minimum horizontal fault position |
| X_Maximum | Maximum horizontal fault position |
| Y_Minimum | Minimum vertical fault position |
| Y_Maximum | Maximum vertical fault position |
| Pixels_Areas | Number/area of pixels associated with the fault |
| X_Perimeter | Horizontal perimeter-related measure |
| Y_Perimeter | Vertical perimeter-related measure |
| Sum_of_Luminosity | Total gray-level intensity |
| Minimum_of_Luminosity | Minimum gray level |
| Maximum_of_Luminosity | Maximum gray level |
| Length_of_Conveyer | Conveyor-related dimensional information |
| Steel_Plate_Thickness | Steel plate thickness |
| Edges_Index | Edge-related descriptor |
| Empty_Index | Shape/occupancy descriptor |
| Square_Index | Shape descriptor |
| Outside_X_Index | Horizontal outside-location descriptor |
| Edges_X_Index | Horizontal edge descriptor |
| Edges_Y_Index | Vertical edge descriptor |
| Outside_Global_Index | Global outside-location descriptor |
| LogOfAreas | Log-transformed area descriptor |
| Log_X_Index | Log-transformed horizontal size descriptor |
| Log_Y_Index | Log-transformed vertical size descriptor |
| Orientation_Index | Fault-orientation descriptor |
| Luminosity_Index | Normalized luminosity descriptor |
| SigmoidOfAreas | Nonlinear transformed area descriptor |

### Important

Not every feature has an immediately obvious physical meaning to a student. That is normal in image-based engineering datasets.

The key is to understand that the features are **measured or computed descriptors of fault geometry and appearance**.

## 9. Install and Import Packages

In [ ]:
!pip -q install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

## 10. Load the Dataset

In [ ]:
steel = fetch_ucirepo(id=198)

X = steel.data.features.copy()
y_raw = steel.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape :", y_raw.shape)

display(X.head())
display(y_raw.head())

### Student question: What should we expect?

Approximately:

- 1,941 rows;
- 27 input features;
- 7 target indicator columns.

## 11. Inspect the Features Before Modeling

In [ ]:
print("Feature names:")
print(X.columns.tolist())

print("\nMissing values:")
print(X.isna().sum())

print("\nData types:")
print(X.dtypes)

display(X.describe().T)

### Why inspect the data first?

Before training a model, we should confirm:

- dimensions are correct;
- values are numerical;
- missing data are understood;
- features vary across observations;
- scales differ substantially.

Different numerical scales will matter when we use logistic regression.

## 12. Convert Seven Indicator Columns into One Class Label

The target is stored in **one-hot / indicator form**.

Example:

| Pastry | Z_Scratch | K_Scatch | ... |
|---:|---:|---:|---|
| 0 | 1 | 0 | ... |

This means the observation belongs to **Z_Scratch**.

We can convert each row to the name of its active class.

In [ ]:
y = y_raw.idxmax(axis=1)

print(y.head())
print("\nNumber of classes:", y.nunique())
print("\nClass names:")
print(sorted(y.unique()))

### Student question: What does idxmax(axis=1) do?

For each row, it finds the column name containing the largest value.

Because the targets are indicators, that column is the active fault class.

## 13. Check Class Balance

In [ ]:
class_counts = y.value_counts()

display(class_counts)

plt.figure(figsize=(9,4))
class_counts.plot(kind="bar")
plt.ylabel("Number of Fault Instances")
plt.xlabel("Fault Class")
plt.title("Steel Fault Class Distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 14. Why Class Imbalance Matters

A dataset is **imbalanced** when some classes occur much more often than others.

Suppose:

- 80% of observations belong to one common class;
- only 2% belong to a rare but dangerous fault.

A model could achieve apparently high accuracy while performing poorly on the rare class.

### Student question: Is imbalance automatically a problem?

Not always.

It becomes important when:

- minority classes matter;
- the model ignores rare classes;
- overall accuracy hides poor per-class performance.

That is why we examine **precision, recall, F1, and the confusion matrix**.

## 15. Train/Test Split

We use:

- 80% training;
- 20% testing;
- random_state=42;
- stratify=y.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).sort_index())

## 16. Why Do We Use stratify=y?

Stratification tries to preserve approximately the same class proportions in both training and test sets.

### Student question: What could happen without stratification?

A rare class could be underrepresented in the test set or training set, especially in smaller datasets.

That would make evaluation less reliable.

## 17. Why Feature Scaling Matters

Features in this dataset have very different numerical ranges.

For example:

- one feature may be near 0.1;
- another may be in the hundreds;
- another may be in the thousands.

Standardization transforms each feature approximately as:

**z = (x − mean) / standard deviation**

This puts features on comparable scales.

For logistic regression, scaling often improves numerical stability and optimization.

## 18. Build a Scaling + Logistic Regression Pipeline

A pipeline ensures scaling is learned from the training data and then applied consistently to test data.

In [ ]:
clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000
    ))
])

clf

### Why use a Pipeline?

Without a pipeline, students may accidentally:

1. scale the entire dataset before splitting;
2. allow information from the test set to influence preprocessing;
3. apply inconsistent transformations.

A pipeline helps prevent this kind of leakage.

## 19. Train the Classifier

In [ ]:
clf.fit(X_train, y_train)

### What does fit() learn here?

Two things happen inside the pipeline:

1. `StandardScaler` learns training-set means and standard deviations.
2. `LogisticRegression` learns coefficients that help separate the classes.

The test data remain unseen during fitting.

## 20. Make Predictions

In [ ]:
y_pred = clf.predict(X_test)

comparison = pd.DataFrame({
    "Actual_Class": y_test.to_numpy(),
    "Predicted_Class": y_pred
})

display(comparison.head(15))

## 21. Accuracy

Accuracy is:

**number of correct predictions / total predictions**

It answers:

> What fraction of test observations did the classifier label correctly?

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy = {accuracy:.3f}")

### Student question: Is accuracy enough?

No.

If one class is very common, high accuracy can hide poor performance on rare classes.

For engineering fault detection, rare faults may be especially important.

## 22. Confusion Matrix

A confusion matrix shows **which classes are confused with which other classes**.

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
    ax=ax
)

plt.title("Steel Fault Classification — Confusion Matrix")
plt.tight_layout()
plt.show()

### How to read a multiclass confusion matrix

- Rows = actual classes
- Columns = predicted classes
- Diagonal cells = correct predictions
- Off-diagonal cells = mistakes

Large off-diagonal values reveal which fault types the model tends to confuse.

## 23. True Positive, False Positive, False Negative

These ideas are easiest to understand by focusing on one class at a time.

Suppose the class of interest is **Z_Scratch**.

- **True Positive (TP):** actual Z_Scratch, predicted Z_Scratch
- **False Positive (FP):** not Z_Scratch, but predicted Z_Scratch
- **False Negative (FN):** actual Z_Scratch, but predicted another class
- **True Negative (TN):** not Z_Scratch and not predicted Z_Scratch

### Engineering meaning of a false negative

A true fault exists, but the system fails to identify it as that fault.

Depending on the application, this could be more serious than a false alarm.

## 24. Precision

Precision asks:

> Of all samples predicted as a particular fault class, how many were actually that class?

**Precision = TP / (TP + FP)**

High precision means relatively few false alarms for that class.

## 25. Recall

Recall asks:

> Of all actual samples belonging to a fault class, how many did the model detect?

**Recall = TP / (TP + FN)**

High recall means relatively few missed cases.

## 26. F1 Score

F1 combines precision and recall:

**F1 = 2 × (Precision × Recall) / (Precision + Recall)**

It is useful when both false positives and false negatives matter.

## 27. Classification Report

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)

### What do support, macro avg, and weighted avg mean?

**Support** = number of true test observations in that class.

**Macro average** = simple average across classes. Every class gets equal weight.

**Weighted average** = average weighted by class frequency. Common classes influence it more.

### Student question: Why might macro F1 be lower than weighted F1?

Because the classifier may perform worse on minority classes. Macro averaging exposes that problem more strongly.

## 28. Calculate Overall Macro Metrics Explicitly

In [ ]:
macro_precision = precision_score(
    y_test, y_pred, average="macro", zero_division=0
)

macro_recall = recall_score(
    y_test, y_pred, average="macro", zero_division=0
)

macro_f1 = f1_score(
    y_test, y_pred, average="macro", zero_division=0
)

print(f"Accuracy        = {accuracy:.3f}")
print(f"Macro Precision = {macro_precision:.3f}")
print(f"Macro Recall    = {macro_recall:.3f}")
print(f"Macro F1        = {macro_f1:.3f}")

## 29. Inspect Misclassified Samples

A model metric tells us **how often** errors occur.

Inspecting rows tells us **where** they occur.

In [ ]:
test_results = X_test.copy()
test_results["Actual_Class"] = y_test
test_results["Predicted_Class"] = y_pred

errors = test_results[
    test_results["Actual_Class"] != test_results["Predicted_Class"]
]

print("Number of misclassified test samples:", len(errors))
display(errors.head(10))

### Engineering question

Are the errors concentrated among specific fault pairs?

That may suggest:

- overlapping feature patterns;
- insufficiently informative features;
- model limitations;
- noisy labels;
- need for a more flexible classifier.

## 30. Predict Class Probabilities

Logistic regression can estimate a probability-like score for each class.

In [ ]:
probabilities = clf.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilities,
    columns=clf.classes_,
    index=X_test.index
)

display(prob_df.head())

## 31. Confidence of the Predicted Class

The maximum predicted probability can help identify uncertain predictions.

In [ ]:
confidence = prob_df.max(axis=1)

confidence_table = pd.DataFrame({
    "Actual_Class": y_test,
    "Predicted_Class": y_pred,
    "Predicted_Confidence": confidence
})

display(
    confidence_table.sort_values(
        "Predicted_Confidence"
    ).head(10)
)

### Important caution

A predicted probability is **not automatically a perfectly calibrated probability**.

A model can be overconfident or underconfident.

Later courses may cover probability calibration in more depth.

## 32. Common Student Questions

### Why is Logistic Regression called regression if we use it for classification?

Because it models a linear combination of features internally, but transforms that result into class probabilities. Despite its name, it is widely used as a classifier.

### Why do we scale before logistic regression?

Because the features have different numerical scales, and scaling often makes optimization more stable.

### Why don't we scale y?

The target contains class labels, not continuous numerical measurements.

### Why use stratify?

To maintain similar class proportions in training and test sets.

### Why can accuracy be high but recall for one class be low?

Because accuracy summarizes all observations. A minority class can perform poorly without strongly changing overall accuracy.

### Is a false positive always less serious than a false negative?

No. The relative cost depends on the engineering system.

### Can the model predict a fault class it never saw during training?

No. Supervised classifiers learn only from the classes represented in the training data.

## 33. Common Mistakes

### Mistake 1 — Treating class labels like continuous numbers

Categories should not be interpreted as numerical magnitudes.

### Mistake 2 — Scaling before the train/test split

This can introduce information leakage.

### Mistake 3 — Looking only at accuracy

Always inspect class-specific performance.

### Mistake 4 — Ignoring class imbalance

Rare but important faults may be missed.

### Mistake 5 — Confusing precision and recall

Precision asks about predicted positives.

Recall asks about actual positives.

### Mistake 6 — Assuming predicted probabilities are perfectly trustworthy

Probability calibration may be imperfect.

### Mistake 7 — Ignoring the engineering cost of errors

A model metric alone does not determine whether a classifier is acceptable.

## 34. Guided Practice

Complete these with your instructor.

1. Identify the most common fault class.
2. Identify the least common class.
3. Find the class with the lowest recall.
4. Find the class with the lowest precision.
5. Find the class with the lowest F1 score.
6. Identify one pair of classes frequently confused in the confusion matrix.
7. Find the five least-confident predictions.
8. Explain whether those uncertain predictions are also misclassifications.
9. Write an engineering interpretation of one false negative.
10. Explain why macro F1 is useful for this dataset.

## 35. Independent Exercises

1. Re-run the split with `random_state=1`. Compare accuracy and macro F1.
2. Change the test size from 20% to 30%.
3. Remove `StandardScaler` and compare results.
4. Train logistic regression with `class_weight="balanced"`.
5. Compare macro recall before and after class weighting.
6. Train a K-nearest-neighbors classifier.
7. Compare KNN and logistic regression.
8. Identify which model gives better macro F1.
9. Create a DataFrame of all misclassified observations.
10. Write a short recommendation about whether overall accuracy alone is sufficient for this quality-control problem.

## 36. Engineering Scenario Discussion

Suppose a particular surface fault could cause downstream product failure.

Would you prioritize:

- accuracy,
- precision,
- recall,
- or F1?

There is no universal answer.

If missing that fault is dangerous, **recall** may become especially important because false negatives are costly.

If false alarms stop production and are extremely expensive, **precision** may also matter strongly.

Engineering evaluation must connect model metrics to real consequences.

## 37. Review Quiz

1. What is classification?
2. What is the difference between binary and multiclass classification?
3. What does one row in the steel dataset represent?
4. How many fault classes are present?
5. Why do we use stratification?
6. Why do we scale features?
7. What does logistic regression output internally?
8. Define accuracy.
9. Define precision.
10. Define recall.
11. Define F1.
12. What is a false negative?
13. Why can accuracy be misleading?
14. What does support mean in a classification report?
15. What is the difference between macro and weighted averages?
16. Why inspect a confusion matrix?
17. What can low-confidence predictions reveal?
18. Why are predicted probabilities not automatically perfectly calibrated?
19. Why is class imbalance important?
20. Why must engineering consequences be considered when choosing a metric?

## 38. Key Takeaways

The Week 8 workflow is:

**Engineering question → dataset → class labels → class balance → stratified split → scaling → classifier → predictions → confusion matrix → precision/recall/F1 → error analysis → engineering interpretation**

The goal is not simply to obtain a high accuracy number.

The goal is to understand **which faults are detected, which faults are missed, what kinds of mistakes occur, and whether those mistakes are acceptable for the engineering application**.